# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR² Mental Health Survey dataset from Kilifi County using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Accessing dataset metadata attributes
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID: {metadata.id}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant datasets, entities (record sets, fields, columns) are referenced by their `@id` attribute. Let's list the available record sets and their fields.

In [ ]:
# List available record sets and fields with their @id
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- Record Set Name: {rs.name} | @id: {rs.id}")
    # List fields for each record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field Name: {field.name} | @id: {field.id} | DataType: {field.data_type}")
    print()
# Save the first record set @id for demonstration
if record_sets:
    record_set_id = record_sets[0].id
else:
    record_set_id = None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll extract the survey data using the record set's `@id` and field `@id`s discovered above.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print columns (field @id) of the first record set
if record_set_id:
    print(f"Columns for record set {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
    dataframes[record_set_id].head()
else:
    print("No record sets available in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a numeric field (e.g., the GAD-7 score field) using its `@id`. We'll filter records, normalize the score, and group by another attribute like gender (using its `@id`).

In [ ]:
# Example field @ids based on Croissant schema documentation or the data overview above:
# We'll pretend that the GAD-7 score field's @id is 'cr:gad7_score' and gender's @id is 'cr:gender'.
# Please update these IDs if your dataset fields differ.
numeric_field_id = 'cr:gad7_score'
group_field_id = 'cr:gender'

df = dataframes.get(record_set_id, pd.DataFrame())

# Filter for GAD-7 scores greater than 10
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize GAD-7 scores
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by gender
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in DataFrame columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we'll plot the distribution of GAD-7 scores, and compare mean score by gender, using field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot GAD-7 score distribution
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title("Distribution of GAD-7 Scores")
    plt.xlabel("GAD-7 Score (@id: cr:gad7_score)")
    plt.ylabel("Frequency")
    plt.show()

# Bar plot of mean GAD-7 by gender
if group_field_id in df.columns:
    mean_scores = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    plt.figure(figsize=(6,3))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=mean_scores)
    plt.title("Mean GAD-7 Score by Gender")
    plt.xlabel("Gender (@id: cr:gender)")
    plt.ylabel("Mean GAD-7 Score")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides comprehensive survey records from Kilifi County, Kenya, with standard metadata accessible via Croissant schema and unique `@id` fields for each entity.
- EDA and visualization reveal distributions of mental health indicators, e.g., GAD-7, and demographic breakdowns.
- The `mlcroissant` library streamlines access to FAIR-compliant datasets for further research and analytics.